# 本地图片（单张/文件夹）→ 4 子图 → VGGT 批量重建 GLB

这个 notebook 用于：
1. 支持 `input_path` 为单张图片或图片文件夹
2. 每张图按 512×512 处理并裁成 4 张 256×256 子图
3. 用 VGGT 推理并生成 `.glb`
4. 结果按原图文件名对应保存到 `output_path`

## 1) 安装与导入依赖（PyTorch / OpenCV / NumPy / Trimesh）

In [1]:
# 如环境已安装可注释掉本单元
# %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install -q opencv-python numpy trimesh ipywidgets

import os
import sys
import glob
import gc
import time
import tempfile
from pathlib import Path

import cv2
import torch
import numpy as np
import trimesh

from IPython.display import display, HTML, FileLink
import matplotlib.pyplot as plt

# 关键：保证可导入 vggt 代码
# 如果你的 notebook 不在 /home/viplab/liukejia/project/vggt 下，请改这里
PROJECT_ROOT = Path('/home/viplab/liukejia/project/vggt').resolve()
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / 'vggt'))

from visual_util import predictions_to_glb
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

print('PROJECT_ROOT =', PROJECT_ROOT)

/home/viplab/anaconda3/envs/vggt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT = /home/viplab/liukejia/project/vggt


## 0) 参数配置（最前设置）

In [2]:
# ===== 必填参数 =====
# 可填单张图片路径，或文件夹路径（会处理文件夹内所有图片）
# input_path = '/home/viplab/liukejia/project/mv_model/results_4in1_sdxl_warp_ckpt13000'
# input_path = '/home/viplab/liukejia/project/mv_model/results_4in1_sdxl_warp_ckpt_21500'
input_path = '/home/viplab/liukejia/project/result/inference_images'
# 输出目录：每张图会生成同名 .glb（以及可选 .npz）
output_path = '/home/viplab/liukejia/project/vggt/outputs_glb_grid3'

# ===== 处理参数 =====
force_resize = True  # 非 512x512 是否强制缩放到 512x512
conf_thres = 50.0
prediction_mode = 'Depthmap and Camera Branch'
save_npz = False

# 支持的图片后缀
valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('input_path =', input_path)
print('output_path =', output_path)

input_path = /home/viplab/liukejia/project/result/inference_images
output_path = /home/viplab/liukejia/project/vggt/outputs_glb_grid3


## 2) 加载 VGGT 模型与权重到 CUDA

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

print('Initializing VGGT...')
model = VGGT()
_URL = 'https://huggingface.co/facebook/VGGT-1B/resolve/main/model.pt'
state_dict = torch.hub.load_state_dict_from_url(_URL, map_location='cpu')
model.load_state_dict(state_dict)
model.eval().to(device)

# 半精度配置（仅 GPU 有意义）
if torch.cuda.is_available():
    major_cc = torch.cuda.get_device_capability()[0]
    amp_dtype = torch.bfloat16 if major_cc >= 8 else torch.float16
else:
    amp_dtype = None

print('amp_dtype =', amp_dtype)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = cuda
Initializing VGGT...
amp_dtype = torch.bfloat16


## 3) 读取 `input_path`（单图或文件夹）并做尺寸校验

In [4]:
def collect_input_images(input_path: str, valid_exts=None):
    if valid_exts is None:
        valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

    p = Path(input_path)
    if not p.exists():
        raise FileNotFoundError(f'input_path not found: {p}')

    if p.is_file():
        if p.suffix.lower() not in valid_exts:
            raise ValueError(f'Unsupported file extension: {p.suffix}')
        return [p]

    if p.is_dir():
        files = [x for x in sorted(p.iterdir()) if x.is_file() and x.suffix.lower() in valid_exts]
        if len(files) == 0:
            raise ValueError(f'No image files found in folder: {p}')
        return files

    raise ValueError(f'Invalid input_path: {p}')


def read_single_image_512(image_path: str, force_resize: bool = True):
    image_path = Path(image_path)
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f'Failed to read image: {image_path}')

    h, w = bgr.shape[:2]
    if (h, w) != (512, 512):
        if force_resize:
            bgr = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)
            print(f'[Info] resized {image_path.name}: {(w, h)} -> (512, 512)')
        else:
            raise ValueError(f'{image_path.name}: expect 512x512, got {(w, h)}')

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return rgb

input_images = collect_input_images(input_path, valid_exts=valid_exts)
print(f'Found {len(input_images)} image(s).')
print('First 5:', [p.name for p in input_images[:5]])

Found 9 image(s).
First 5: ['num123_-800_100_0_0007.png', 'num123_-800_150_0_0008.png', 'num123_-800_200_0_0009.png', 'num123_-850_100_0_0004.png', 'num123_-850_150_0_0005.png']


## 4) 将单张图裁成 4 个 256×256 子图并写入 `target_dir/images`

In [5]:
def split_512_to_4x256_and_save(img_rgb: np.ndarray, target_dir: str):
    target_dir = Path(target_dir)
    images_dir = target_dir / 'images'
    images_dir.mkdir(parents=True, exist_ok=True)

    # 固定顺序：左上、右上、左下、右下
    patches = [
        img_rgb[0:256, 0:256],      # 000000
        img_rgb[0:256, 256:512],    # 000001
        img_rgb[256:512, 0:256],    # 000002
        img_rgb[256:512, 256:512],  # 000003
    ]

    patch_paths = []
    for i, p in enumerate(patches):
        out_path = images_dir / f'{i:06d}.png'
        cv2.imwrite(str(out_path), cv2.cvtColor(p, cv2.COLOR_RGB2BGR))
        patch_paths.append(str(out_path))

    cv2.imwrite(str(target_dir / 'original_512.png'), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    return patch_paths

# 仅快速检查第一张（可选）
_preview_rgb = read_single_image_512(str(input_images[0]), force_resize=force_resize)
_preview_dir = Path(tempfile.mkdtemp(prefix='vggt_preview_'))
_ = split_512_to_4x256_and_save(_preview_rgb, _preview_dir)
print('preview target_dir =', _preview_dir)

preview target_dir = /tmp/vggt_preview_0m4k0n3z


## 5) 运行 VGGT 推理并生成 `extrinsic/intrinsic/world_points`

In [6]:
def run_model(target_dir: str, model) -> dict:
    target_dir = Path(target_dir)
    image_names = sorted(glob.glob(str(target_dir / 'images' / '*')))
    if len(image_names) == 0:
        raise ValueError(f'No images found in {target_dir / "images"}')

    images = load_and_preprocess_images(image_names).to(device)

    with torch.no_grad():
        if torch.cuda.is_available():
            with torch.cuda.amp.autocast(dtype=amp_dtype):
                predictions = model(images)
        else:
            predictions = model(images)

    extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions['pose_enc'], images.shape[-2:])
    predictions['extrinsic'] = extrinsic
    predictions['intrinsic'] = intrinsic

    for key in list(predictions.keys()):
        if isinstance(predictions[key], torch.Tensor):
            predictions[key] = predictions[key].detach().cpu().numpy().squeeze(0)

    predictions['pose_enc_list'] = None

    depth_map = predictions['depth']
    world_points = unproject_depth_map_to_point_map(
        depth_map,
        predictions['extrinsic'],
        predictions['intrinsic']
    )
    predictions['world_points_from_depth'] = world_points

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return predictions

## 6) 将预测结果按图片名导出到 `output_path`（`.glb` / 可选 `.npz`）

In [7]:
def export_npz_and_glb(
    sample_target_dir: str,
    predictions: dict,
    output_dir: str,
    output_stem: str,
    conf_thres: float = 50.0,
    prediction_mode: str = 'Depthmap and Camera Branch',
    save_npz: bool = True,
):
    sample_target_dir = Path(sample_target_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    glb_path = output_dir / f'{output_stem}.glb'
    npz_path = output_dir / f'{output_stem}.npz'

    if save_npz:
        np.savez(npz_path, **predictions)

    glbscene = predictions_to_glb(
        predictions,
        conf_thres=conf_thres,
        filter_by_frames='All',
        show_cam=True,
        mask_sky=False,
        mask_black_bg=False,
        mask_white_bg=False,
        target_dir=str(sample_target_dir),
        prediction_mode=prediction_mode,
    )
    glbscene.export(file_obj=str(glb_path))

    return (str(npz_path) if save_npz else None), str(glb_path)

## 7) 批量执行：支持 `input_path` 为单图或文件夹，结果保存到 `output_path`

In [8]:
def reconstruct_one_image(
    image_path: Path,
    output_dir: Path,
    conf_thres: float,
    prediction_mode: str,
    force_resize: bool,
    save_npz: bool,
):
    img_rgb = read_single_image_512(str(image_path), force_resize=force_resize)

    # 每张图独立临时目录，避免相互覆盖
    sample_target_dir = Path(tempfile.mkdtemp(prefix=f'vggt_{image_path.stem}_'))
    _ = split_512_to_4x256_and_save(img_rgb, sample_target_dir)

    predictions = run_model(sample_target_dir, model)
    npz_path, glb_path = export_npz_and_glb(
        sample_target_dir=sample_target_dir,
        predictions=predictions,
        output_dir=output_dir,
        output_stem=image_path.stem,
        conf_thres=conf_thres,
        prediction_mode=prediction_mode,
        save_npz=save_npz,
    )

    return {
        'input_image': str(image_path),
        'target_dir': str(sample_target_dir),
        'glb_path': glb_path,
        'npz_path': npz_path,
    }


def reconstruct_from_input_path(
    input_path: str,
    output_path: str,
    conf_thres: float = 50.0,
    prediction_mode: str = 'Depthmap and Camera Branch',
    force_resize: bool = True,
    save_npz: bool = True,
    valid_exts=None,
):
    t0 = time.time()
    output_dir = Path(output_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    images = collect_input_images(input_path, valid_exts=valid_exts)
    print(f'[Start] total images: {len(images)}')

    results = []
    for i, img_path in enumerate(images, 1):
        print(f'[{i}/{len(images)}] processing: {img_path.name}')
        item = reconstruct_one_image(
            image_path=img_path,
            output_dir=output_dir,
            conf_thres=conf_thres,
            prediction_mode=prediction_mode,
            force_resize=force_resize,
            save_npz=save_npz,
        )
        results.append(item)

    elapsed = time.time() - t0
    print(f'[Done] {len(results)} image(s), elapsed: {elapsed:.2f}s')
    print('output_dir =', output_dir)
    return results

## 8) 一键运行（只保存，不显示 GLB）

In [9]:
results = reconstruct_from_input_path(
    input_path=input_path,
    output_path=output_path,
    conf_thres=conf_thres,
    prediction_mode=prediction_mode,
    force_resize=force_resize,
    save_npz=save_npz,
    valid_exts=valid_exts,
)

print('\n生成结果（前10条）:')
for row in results[:10]:
    print(f"- {Path(row['input_image']).name} -> {Path(row['glb_path']).name}")

# 结果会保存到 output_path，命名与输入图片同名（后缀改为 .glb / .npz）
results[:3]

[Start] total images: 9
[1/9] processing: num123_-800_100_0_0007.png
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[2/9] processing: num123_-800_150_0_0008.png
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[3/9] processing: num123_-800_200_0_0009.png
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[4/9] processing: num123_-850_100_0_0004.png
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[5/9] processing: num123_-850_150_0_0005.png
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[6/9] processing: num123_-850_200_0_0006.png
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[7/9] processing: num123_-900_100_0_0001.png
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[8/9] processing: num123_-900_150_0_0002.png
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[9/9] processing: num123_-900_200_0_0003.png
Building GLB scene
Using De

[{'input_image': '/home/viplab/liukejia/project/result/inference_images/num123_-800_100_0_0007.png',
  'target_dir': '/tmp/vggt_num123_-800_100_0_0007_eoestwf8',
  'glb_path': '/home/viplab/liukejia/project/vggt/outputs_glb_grid3/num123_-800_100_0_0007.glb',
  'npz_path': None},
 {'input_image': '/home/viplab/liukejia/project/result/inference_images/num123_-800_150_0_0008.png',
  'target_dir': '/tmp/vggt_num123_-800_150_0_0008_mpr6w9ga',
  'glb_path': '/home/viplab/liukejia/project/vggt/outputs_glb_grid3/num123_-800_150_0_0008.glb',
  'npz_path': None},
 {'input_image': '/home/viplab/liukejia/project/result/inference_images/num123_-800_200_0_0009.png',
  'target_dir': '/tmp/vggt_num123_-800_200_0_0009_0hvgmaxr',
  'glb_path': '/home/viplab/liukejia/project/vggt/outputs_glb_grid3/num123_-800_200_0_0009.glb',
  'npz_path': None}]